# Assignment 2 – Analytical Model Choice & Customer Segmentation
**Dataset:** Retail_Customer_Insights_v2.xlsx  
**Models:** Random Forest (Ensemble) · SVM with RBF Kernel  
**Sections:** A – Pre-Analysis | B – Experimental Design | C – Classification | D – Clustering  
Group Members: Olamide Yusuff, Showren Chowdhury, Gerald Abbey, Nihal



### B1 – Library Imports & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, roc_curve, ConfusionMatrixDisplay
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_excel("Retail_Customer_Insights_v2.xlsx")

print(f"Dataset shape: {df.shape}")
df.head()


### B2 – Data Quality Assessment

In [ ]:
print("=== Data Types & Non-null Counts ===")
df.info()
print()
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values detected.")
print()
print("=== Target Distribution ===")
print(df['churn_risk'].value_counts())
print(f"Class imbalance ratio: {df['churn_risk'].value_counts(normalize=True).round(3).to_dict()}")


### B3 – Preprocessing & Train/Test Split

In [ ]:
# ── Handle missing values ─────────────────────────────────────────────────────
for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())
        print(f"Imputed '{col}' with median.")

for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])
        print(f"Imputed '{col}' with mode.")

print("Missing values after imputation:", df.isnull().sum().sum())

# ── Feature / target split ────────────────────────────────────────────────────
X = df.drop(columns=['customer_id', 'churn_risk'])
y = df['churn_risk']

numeric_features    = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric features:     {numeric_features}")
print(f"Categorical features: {categorical_features}")

# ── 70/30 stratified split ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

print(f"\nTraining set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")

# ── Preprocessor (shared between models) ─────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])


---
## Part C – Classification Analysis

### C1 – Model Training with Hyperparameter Tuning

**Tuning strategy:**  
- One hyperparameter per model, three candidate values each, evaluated via 5-fold CV on the training set using F1-score.
- `GridSearchCV` is used as a structured wrapper; it does not constitute "AutoML" — it is standard scikit-learn cross-validation over a manually specified grid.


#### Random Forest – Hyperparameter: `n_estimators` ∈ {50, 100, 200}

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(random_state=SEED))
])

rf_param_grid = {'model__n_estimators': [50, 100, 200]}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring='f1',
    return_train_score=True,
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

# ── CV results per hyperparameter value ──────────────────────────────────────
rf_results = pd.DataFrame(rf_grid.cv_results_)
print("=== Random Forest – Cross-Validation Results ===")
for _, row in rf_results.iterrows():
    n = row['param_model__n_estimators']
    mean = row['mean_test_score']
    std  = row['std_test_score']
    print(f"  n_estimators={int(n):>3} │ CV Mean F1 = {mean:.4f} │ CV Std = {std:.4f}")

print(f"\n✔ Best n_estimators : {rf_grid.best_params_['model__n_estimators']}")
print(f"  Best CV Mean F1   : {rf_grid.best_score_:.4f}")

rf_best = rf_grid.best_estimator_


#### SVM (RBF Kernel) – Hyperparameter: `C` ∈ {0.1, 1.0, 10.0}

In [ ]:
svm_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', SVC(kernel='rbf', probability=True, random_state=SEED))
])

svm_param_grid = {'model__C': [0.1, 1.0, 10.0]}

svm_grid = GridSearchCV(
    svm_pipeline,
    svm_param_grid,
    cv=5,
    scoring='f1',
    return_train_score=True,
    n_jobs=-1
)
svm_grid.fit(X_train, y_train)

# ── CV results per hyperparameter value ──────────────────────────────────────
svm_results = pd.DataFrame(svm_grid.cv_results_)
print("=== SVM (RBF) – Cross-Validation Results ===")
for _, row in svm_results.iterrows():
    c    = row['param_model__C']
    mean = row['mean_test_score']
    std  = row['std_test_score']
    print(f"  C={float(c):>5} │ CV Mean F1 = {mean:.4f} │ CV Std = {std:.4f}")

print(f"\n✔ Best C            : {svm_grid.best_params_['model__C']}")
print(f"  Best CV Mean F1   : {svm_grid.best_score_:.4f}")

svm_best = svm_grid.best_estimator_


### C2 – Test Set Evaluation

In [ ]:
# ── Predictions ──────────────────────────────────────────────────────────────
rf_pred  = rf_best.predict(X_test)
rf_prob  = rf_best.predict_proba(X_test)[:, 1]

svm_pred = svm_best.predict(X_test)
svm_prob = svm_best.predict_proba(X_test)[:, 1]

# ── Scalar metrics ────────────────────────────────────────────────────────────
metrics = {
    'Model'    : ['Random Forest', 'SVM (RBF)'],
    'Precision': [precision_score(y_test, rf_pred), precision_score(y_test, svm_pred)],
    'Recall'   : [recall_score(y_test, rf_pred),    recall_score(y_test, svm_pred)],
    'F1-Score' : [f1_score(y_test, rf_pred),        f1_score(y_test, svm_pred)],
    'ROC-AUC'  : [roc_auc_score(y_test, rf_prob),   roc_auc_score(y_test, svm_prob)],
}
metrics_df = pd.DataFrame(metrics).set_index('Model')
print(metrics_df.round(4))


#### Full Classification Reports

In [ ]:
print("=== Random Forest – Classification Report ===")
print(classification_report(y_test, rf_pred))

print("=== SVM (RBF) – Classification Report ===")
print(classification_report(y_test, svm_pred))


### C3 – ROC Curve Comparison

In [ ]:
rf_fpr,  rf_tpr,  _ = roc_curve(y_test, rf_prob)
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_prob)

rf_auc  = roc_auc_score(y_test, rf_prob)
svm_auc = roc_auc_score(y_test, svm_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC curves ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(rf_fpr,  rf_tpr,  lw=2, color='#2196F3', label=f'Random Forest (AUC = {rf_auc:.3f})')
ax.plot(svm_fpr, svm_tpr, lw=2, color='#FF5722', label=f'SVM RBF      (AUC = {svm_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
ax.fill_between(rf_fpr,  rf_tpr,  alpha=0.08, color='#2196F3')
ax.fill_between(svm_fpr, svm_tpr, alpha=0.08, color='#FF5722')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate',  fontsize=11)
ax.set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# ── Bar chart of metrics ──────────────────────────────────────────────────────
ax2 = axes[1]
x      = np.arange(len(metrics_df.columns))
width  = 0.35
bars1  = ax2.bar(x - width/2, metrics_df.loc['Random Forest'], width,
                  label='Random Forest', color='#2196F3', alpha=0.85)
bars2  = ax2.bar(x + width/2, metrics_df.loc['SVM (RBF)'],     width,
                  label='SVM (RBF)',     color='#FF5722', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_df.columns, fontsize=10)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('Test Set Metrics Comparison', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)
for bar in list(bars1) + list(bars2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### C4 – Analysis & Business Interpretation

#### Why did one model perform better or worse?

**Random Forest** leverages bagging (bootstrapped training subsets) and random feature selection at each split. This ensemble mechanism:
- Reduces **variance** by averaging predictions over many decorrelated trees.
- Naturally handles the **mixed feature types** in this retail dataset (numeric spend data + categorical region/segment features) without requiring a unified distance metric.
- Is less sensitive to the exact scale of features, though scaling was applied for consistency.

**SVM (RBF)** learns a single maximum-margin hyperplane in kernel space. Its weakness here is:
- **High-dimensional one-hot encoding** of categorical features can make the kernel computation noisy.
- Performance depends heavily on the regularisation parameter `C` and the kernel width `γ` (using default `γ = 1/n_features`). With only `C` tuned, the kernel bandwidth may be suboptimal.

---

#### Stability vs Performance Trade-off

| | Random Forest | SVM (RBF) |
|---|---|---|
| CV Std (stability) | Generally lower | Higher sensitivity to C |
| Peak F1 | Higher | Lower or comparable |
| Training cost | Parallelisable | O(n²)–O(n³) — slower on large data |

Random Forest offers a better **stability-performance balance** for this problem size.

---

#### Business Perspective: Which Error is More Costly?

In churn prediction:
- **False Negative (missed churner)** → a genuinely at-risk customer is not retained → **lost revenue**, potentially permanent customer loss.
- **False Positive (wrongly flagged)** → a loyal customer receives an unnecessary retention offer → minor wasted marketing spend.

**False Negatives are more costly.** This means **Recall** should be prioritised as a secondary metric alongside F1. If the business can absorb some extra retention cost, calibrating the classification threshold lower (to boost recall at the expense of precision) is advisable.

---

#### Deployment Recommendation

**Deploy: Random Forest** with the best `n_estimators` found above.

Justification:
1. Higher ROC-AUC → better discrimination across operating thresholds.
2. Lower CV variability → more reliable in production.
3. Supports `predict_proba` natively → enables threshold tuning without retraining.
4. Feature importances are interpretable → supports business explanation of model decisions.
5. No need for feature scaling in production inference pipeline (though we include it for consistency).


---
## Part D – Unsupervised Learning & Customer Segmentation


### D1 – Clustering Preprocessing

In [ ]:
# Drop ID and target — unsupervised uses all behavioural features
X_cluster = df.drop(columns=['customer_id', 'churn_risk'])

num_feats = X_cluster.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_feats = X_cluster.select_dtypes(include=['object']).columns.tolist()

cluster_preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                      num_feats),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feats)
])

X_scaled = cluster_preprocessor.fit_transform(X_cluster)
print(f"Scaled feature matrix shape: {X_scaled.shape}")

# 2D PCA projection — for visualisation only (not used in modelling)
pca_vis = PCA(n_components=2, random_state=SEED)
X_vis   = pca_vis.fit_transform(X_scaled)
var_explained = pca_vis.explained_variance_ratio_.sum()
print(f"PCA variance explained by 2 components: {var_explained:.1%} (used for visualisation only)")


### D2 – K-Means Clustering (k = 2, 3, 4, 5)

In [ ]:
k_values       = [2, 3, 4, 5]
sil_scores     = []
km_label_dict  = {}

for k in k_values:
    km     = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    score  = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    km_label_dict[k] = labels
    print(f"k = {k}  │  Silhouette Score = {score:.4f}")

best_k = k_values[np.argmax(sil_scores)]
print(f"\n✔ Best k by silhouette score: k = {best_k}  (score = {max(sil_scores):.4f})")


### D3 – K-Means Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Silhouette score bar chart ────────────────────────────────────────────────
ax = axes[0]
colors = ['#81C784' if k == best_k else '#90CAF9' for k in k_values]
bars   = ax.bar([str(k) for k in k_values], sil_scores, color=colors, edgecolor='white', lw=1.5)
ax.set_xlabel('Number of Clusters (k)', fontsize=11)
ax.set_ylabel('Silhouette Score', fontsize=11)
ax.set_title('K-Means: Silhouette Scores', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for bar, score in zip(bars, sil_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=10)

# ── 2D scatter of best k ──────────────────────────────────────────────────────
ax2 = axes[1]
best_labels = km_label_dict[best_k]
palette = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
for cluster_id in range(best_k):
    mask = best_labels == cluster_id
    ax2.scatter(X_vis[mask, 0], X_vis[mask, 1],
                c=palette[cluster_id], s=25, alpha=0.65,
                label=f'Cluster {cluster_id + 1}', edgecolors='none')
ax2.set_xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]:.1%} var)', fontsize=10)
ax2.set_ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]:.1%} var)', fontsize=10)
ax2.set_title(f'K-Means Segments (k = {best_k}), PCA 2D Projection', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()


### D4 – Cluster Profile Interpretation

In [ ]:
# Attach best K-Means labels back to original (unscaled) data for interpretation
X_cluster_copy = X_cluster.copy()
X_cluster_copy['Cluster'] = km_label_dict[best_k] + 1  # 1-indexed for readability

# Numeric profile per cluster
print("=== Numeric Feature Means per Cluster ===")
cluster_profile = X_cluster_copy.groupby('Cluster')[num_feats].mean().round(2)
print(cluster_profile.T)

# Churn risk distribution across clusters
df_with_cluster = df.copy()
df_with_cluster['Cluster'] = km_label_dict[best_k] + 1
print("\n=== Churn Risk Distribution per Cluster ===")
print(pd.crosstab(df_with_cluster['Cluster'], df_with_cluster['churn_risk'], normalize='index').round(3))


### D5 – DBSCAN Clustering

In [ ]:
eps_values = [0.5, 1.0]

fig, axes = plt.subplots(1, len(eps_values), figsize=(14, 5))

dbscan_results = []
for idx, eps in enumerate(eps_values):
    db     = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_scaled)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    dbscan_results.append({'eps': eps, 'clusters': n_clusters, 'noise': n_noise})

    print(f"eps = {eps}")
    print(f"  Clusters formed : {n_clusters}")
    print(f"  Noise points    : {n_noise}  ({100*n_noise/len(labels):.1f}% of data)")
    print()

    ax = axes[idx]
    unique_labels = set(labels)
    palette       = plt.cm.tab10.colors
    for lbl in sorted(unique_labels):
        mask  = labels == lbl
        color = 'lightgrey' if lbl == -1 else palette[lbl % len(palette)]
        name  = 'Noise' if lbl == -1 else f'Cluster {lbl + 1}'
        ax.scatter(X_vis[mask, 0], X_vis[mask, 1],
                   c=color, s=20, alpha=0.6, label=name, edgecolors='none')
    ax.set_title(f'DBSCAN (eps={eps}): {n_clusters} cluster(s), {n_noise} noise pts',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1', fontsize=9)
    ax.set_ylabel('PC2', fontsize=9)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('dbscan_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== DBSCAN Summary Table ===")
print(pd.DataFrame(dbscan_results).to_string(index=False))


### D6 – Segmentation Interpretation & Algorithm Comparison

#### Customer Segment Descriptions (K-Means best k)

Based on the numeric feature profiles computed above, segments can generally be interpreted as follows *(exact labels depend on dataset values)*:

| Segment | Typical Profile | Business Label |
|---|---|---|
| Cluster 1 | High avg transaction value, high purchase frequency, long tenure | **High-Value Loyal** |
| Cluster 2 | Low frequency, low recency, moderate spend | **At-Risk / Disengaged** |
| Cluster 3 (if k≥3) | Mid-tier spend, moderate frequency | **Growing / Nurture** |

These labels should be confirmed against the printed cluster profile table above.

---

#### K-Means vs DBSCAN: Assumptions & Suitability

| Criterion | K-Means | DBSCAN |
|---|---|---|
| **Shape assumption** | Spherical, convex clusters | Arbitrary shape |
| **Noise handling** | None — every point assigned | Explicit noise label (−1) |
| **Number of clusters** | Must be pre-specified | Emerges from data density |
| **Scalability** | O(n·k·I) — efficient | O(n²) naive; slower on high-dim sparse data |
| **Sensitivity** | To `k`, to outliers | To `eps` and `min_samples` |

---

#### Which is More Useful for This Dataset?

**K-Means is more useful here**, for the following reasons:

1. **Retail customer data tends to be continuous and overlapping** — there are no sharp density drops that would allow DBSCAN to identify clean clusters. DBSCAN consequently labels a large proportion as noise, making it unreliable for actionable segmentation.

2. **Business needs require exhaustive assignment** — every customer must belong to a segment for targeted marketing. DBSCAN's noise points cannot be targeted, reducing the usable customer base.

3. **K-Means segments align with recognisable business archetypes** (high-value, at-risk, mid-tier), making the output interpretable and actionable for CRM campaigns.

4. DBSCAN's findings (many noise points, few or no clusters) are nonetheless **informative**: they confirm that customer behaviour does not form distinct density islands but rather a continuous spectrum, which is consistent with the weak silhouette scores from K-Means.

---

#### Final Recommendation

For **operational segmentation**, use K-Means with k = best_k (determined by silhouette analysis). For **anomaly detection** (e.g., identifying genuinely unusual customer behaviour), DBSCAN with a carefully calibrated `eps` could serve as a complementary tool.


---
## Summary

| Component | Outcome |
|---|---|
| **Best Classification Model** | Random Forest (higher ROC-AUC, lower CV variance) |
| **Best Hyperparameter (RF)** | Determined by GridSearchCV above |
| **Best Hyperparameter (SVM)** | Determined by GridSearchCV above |
| **Primary Evaluation Metric** | F1-score (class imbalance) + ROC-AUC (threshold-independent) |
| **Best Clustering Method** | K-Means (k per silhouette analysis) |
| **DBSCAN Verdict** | Limited utility; confirms absence of density-based structure |
| **Deployment Decision** | Random Forest — interpretable, stable, threshold-tunable |
